# Beyond Stated Preferences: Building a Better Match Predictor for Dating Apps

**Dataset:** Columbia Business School Speed Dating Experiment (Fisman & Iyengar)  
**Scope:** 551 participants — 8,378 encounters — 21 experimental waves  
**Author:** Arvind B.  
**Last updated:** March 2026  

---

> *"People don't choose what they say they want.  
> This project quantifies that gap — and builds a better predictor from it."*

---

## Table of Contents

- [Section 0 — Introduction & Business Context](#section-0)
- [Section 1 — Business Framing](#section-1)
- [Section 2 — Exploratory Data Analysis](#section-2)
- [Section 3 — Feature Engineering](#section-3)
- [Section 4 — Modeling](#section-4)
- [Section 5 — Interpretability & Action Levers](#section-5)
- [Section 6 — Business Recommendations](#section-6)

## Section 0 — Introduction & Business Context <a id="section-0"></a>

### 0.1 Project Pitch

Match rates in speed dating events hover around **16.5%** — meaning 8 out of 10 encounters lead nowhere.

Most dating apps try to solve this by asking users what they want: rate the importance of attractiveness, intelligence, shared interests. Then they match people accordingly.

**The problem:** this data suggests people don't actually choose what they say they want.

This project uses the Columbia Speed Dating dataset to:
1. **Quantify the gap** between stated preferences and revealed choices
2. **Measure the self-overestimation bias** — and its impact on match rates
3. **Build a match prediction model** trained on what people *actually* respond to, not what they claim to

The output is a data-driven foundation for a smarter recommendation engine — one that replaces declared preferences with behavioral signals.

### 0.2 Dataset Architecture

The dataset was collected during experimental speed dating events organized by Columbia Business School between 2002 and 2004.

**Structure:** Each row represents **one encounter between two participants** — not one participant.  
If participant A meets participant B, there are two rows: A's perspective and B's perspective.

**Three temporal layers:**

| Layer | Timing | Variables |
|---|---|---|
| Pre-event | Before the night | Stated preferences, demographic info, self-ratings |
| During event | After each 4-minute date | Scores given to each partner (attractiveness, fun, sincerity, intelligence, ambition, shared interests) |
| Post-event | Day after & 3 weeks later | Updated preferences, follow-up dates, satisfaction |

**Key variables:**

| Variable | Description |
|---|---|
| `match` | Target — 1 if both participants said yes to each other |
| `dec` / `dec_o` | Individual decision (yes/no) from each side |
| `attr` / `attr_o` | Attractiveness score given / received |
| `attr1_1` | Stated importance of attractiveness before the event |
| `attr3_1` | Self-rated attractiveness before the event |
| `int_corr` | Correlation of interests between the two participants |
| `samerace` | 1 if both participants share the same racial background |

In [3]:
# ── 0.3 Setup — Imports & Data Loading ────────────────────────────────────

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

print("Libraries loaded successfully")

Libraries loaded successfully


### Project Structure
```
speed-dating-match-prediction/
│
├── data/                          # Raw dataset (not pushed to GitHub)
│   └── Speed_Dating_Data.csv
│
├── notebooks/                     # Main analysis notebook
│   └── speed_dating_analysis.ipynb
│
├── outputs/                       # Charts, tables, model outputs
│
├── src/                           # Helper functions (future)
│
├── presentations/                 # Business presentation (future)
│
└── README.md
```

In [4]:
# Robust path resolution regardless of VS Code working directory
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DATA_PATH = os.path.join(BASE_DIR, "data", "Speed_Dating_Data.csv")

df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")

print(f"Shape: {df.shape}")
print(f"Unique participants: {df['iid'].nunique()}")
print(f"Waves: {df['wave'].nunique()}")
print(f"Mean encounters per participant: {df.groupby('iid').size().mean():.1f}")

Shape: (8378, 195)
Unique participants: 551
Waves: 21
Mean encounters per participant: 15.2


## Section 1 — Business Framing <a id="section-1"></a>

### 1.1 Business Context

A dating app is experiencing a **decline in match rates**.

The product team's hypothesis: the current recommendation engine relies on **stated preferences** — what users say they want when setting up their profile. But if users don't actually choose what they claim to want, the entire matching logic is built on flawed inputs.

**The business stakes:**
- Low match rates → poor user experience → churn
- A 1% improvement in match rate across 1M daily users = 10,000 additional matches per day
- Better matches → longer sessions, higher retention, stronger monetization

**Our role:** acting as the data science team embedded with the product manager, we use the Columbia Speed Dating dataset as a proxy for real app behavior. The experimental design — controlled encounters, pre/during/post measurements — makes it uniquely suited to test the stated vs. revealed preference hypothesis.

### 1.2 The Two Types of Errors and Their Cost

In a binary match prediction model, two types of errors are possible:

| Error | Definition | Business Impact |
|---|---|---|
| **False Negative** | Model predicts no match — but both users would have said yes | Missed connection — user never sees a potentially good match → frustration, churn |
| **False Positive** | Model predicts match — but one or both users say no | Bad recommendation — erodes trust in the app over time |

**Which error is more costly?**

In a dating app context, **false negatives are more damaging** than false positives.  
A user who never gets a good match leaves. A user who gets an occasional bad recommendation stays — and keeps swiping.

**Implication for modeling:**  
We will optimize for **recall** on the positive class (match = 1), accepting a moderate precision trade-off.  
Our primary evaluation metric will be **AUC-ROC**, complemented by **F1-score**.  
Accuracy alone is misleading here — a model that always predicts "no match" achieves 83.5% accuracy while being completely useless.

### 1.3 Stakeholders

| Stakeholder | Role | What they need from this analysis |
|---|---|---|
| **Product Manager** | Owns the recommendation engine roadmap | Clear evidence that stated preferences are poor signals — justification to redesign the matching algorithm |
| **Data Science Team** | Builds and maintains the model | Reliable features, interpretable model, reproducible pipeline |
| **Growth & Retention Team** | Tracks match rates, session length, churn | Quantified impact of improving match prediction on key business metrics |
| **UX Team** | Designs the onboarding flow | Insights on self-overestimation bias — to redesign how users set up their profiles |
| **Users** | End consumers | More relevant matches, less time wasted |

**Primary audience for this analysis:** the Product Manager and the Data Science Team.  
Recommendations in Section 6 are framed accordingly — actionable, specific, tied to product decisions.

### 1.4 The Three Business Questions

This analysis is structured around three questions, each building on the previous one:

| # | Question | Method | Output |
|---|---|---|---|
| **Q1** | Do stated preferences predict actual choices? | Correlation analysis, statistical testing | Quantified gap between declared weights and revealed drivers |
| **Q2** | Do users who overestimate themselves match less? | Self-rating vs. received rating comparison, segmentation by gender | Overestimation score and its impact on individual match rate |
| **Q3** | Can we predict a mutual match from behavioral signals? | Classification model (Logistic Regression + XGBoost) | AUC-ROC, F1, actionable feature importance |

**The logical thread:**  
Q1 demonstrates that the current matching logic is flawed.  
Q2 identifies a specific behavioral bias that compounds the problem.  
Q3 builds a better predictor — trained on what people *actually* respond to.